# Medical Signal Processing — Lab Project
# Multi-Center Breast DCE-MRI: From Raw Images to Harmonized Biomarkers

**Course:** Medical Signal Processing &nbsp;·&nbsp; **Format:** Guided build-up lab (9 parts)

---

In this lab you will rebuild — **one piece at a time** — a complete research pipeline that
analyses breast **Dynamic Contrast-Enhanced MRI (DCE-MRI)** from several hospitals and turns
the raw scans into clean, comparable numerical biomarkers.

This is a *real* pipeline used in oncological imaging research. We have broken it into 9 small
parts. Each part:

1. explains the **idea** behind it (short theory),
2. gives you a **task** ("build this function"),
3. lets you **run and see** the result immediately,
4. ends with **exercises** to test your understanding.

By the end, every function you write is plugged together into the final pipeline in **Part 7**,
and you produce the same outputs (colormaps, CSV feature tables, validation figures) as the
original research code.

> 🧭 **How to use this notebook.** Run the cells from top to bottom. Read the markdown before
> each code cell. Do the exercises. Do **not** skip Part 0 — it creates the data everything else
> needs.



> 📝 **This is the student assignment.** Functions marked `# TODO (Part N)` are **yours to write** — they currently `raise NotImplementedError`. Implement each one from the Task description above it, then run its cell and the check cell below. Work top to bottom: later parts depend on earlier ones. Solutions are intentionally not included here.

---
## Part 0 — The Problem, the Plan, and the Data

### 0.1 The clinical question

After a contrast agent (gadolinium) is injected into a vein, a breast tumour "lights up" on MRI.
*How* the brightness changes over the next minute tells radiologists something about the tumour:

| Pattern | What the signal does after injection | Clinical hint |
|---|---|---|
| **Uptake**  | keeps rising | more often benign |
| **Plateau** | rises then flattens | intermediate |
| **Washout** | rises then **falls** | more often malignant |

We acquire two 3D volumes per patient: **pre-contrast** (`_0000`, before injection) and
**post-contrast** (`_0001`, ~1 min after). A radiologist also gives us a **segmentation mask**
that marks exactly which voxels are tumour (the *Region Of Interest*, ROI).

### 0.2 The multi-center twist

The data comes from **different hospitals with different scanners**. A Siemens 3T machine and a
GE 1.5T machine produce systematically different numbers *even for an identical tumour*. If we
just pool all hospitals together, those scanner differences ("**batch effects**") contaminate
the analysis. The last part of the pipeline removes them with **harmonization**.

### 0.3 The pipeline you will build

```
 raw NIfTI  ─►  explore   ─►  kinetic      ─►  pseudo-color  ─►  RGB NIfTI
 (Part 1)       & QC          analysis         visualization     export
                              (Part 2)         (Part 3)          (Part 4)
                                  │
                                  ▼
                            radiomics features ──► feature table ──► harmonization ──► validation
                              (Part 5)              (Part 6)          (Part 6)          (Part 8)
                                                         └────────── assembled in Part 7 ──────────┘
```

### 0.4 Setup

Run the next cell. It only needs the standard scientific Python stack plus `nibabel`
(the library for reading `.nii.gz` medical images).

In [ ]:
# --- Core libraries (all standard, pip-installable) ---
import os, sys, glob, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

warnings.filterwarnings("ignore")

try:
    import nibabel as nib
except ImportError:
    print("nibabel is missing. Install it with:  pip install nibabel")
    raise

np.random.seed(42)                 # reproducible results for everyone
plt.rcParams["figure.dpi"] = 110

print("Environment ready.")
print("numpy", np.__version__, "| pandas", pd.__version__, "| nibabel", nib.__version__)

### 0.5 Create a synthetic multi-center dataset

The real MAMA-MIA hospital data cannot be shared (patient privacy + file size). So we **build a
tiny phantom dataset ourselves** that behaves like the real thing:

* a small 3D volume with a **tumour blob**,
* a **pre** and a **post** contrast version (the tumour enhances),
* a **mask** of the tumour,
* every tumour has its **own kinetic composition** (some uptake-dominant, some
  washout-dominant), so the per-case statistics in Part 9 are meaningful,
* **two "hospitals"** (`CENTER_A`, `CENTER_B`) whose scanners apply a different *gain* and
  *offset* and see slightly different tumour mixes — the batch effect we later remove.

We save everything on disk using **exactly the folder layout of the original project** so the
code you write here would work unchanged on the real data:

```
mamamia_data/
  CENTER_A/
    CENTER_A_001/  CENTER_A_001_0000.nii.gz   CENTER_A_001_0001.nii.gz
    segment/       CENTER_A_001.nii.gz
  CENTER_B/ ...
```

In [ ]:
DATA_ROOT = "mamamia_data"
CENTERS   = ["CENTER_A", "CENTER_B"]
N_CASES   = 4                       # cases per center
VOL_SHAPE = (64, 64, 24)            # small on purpose so the lab runs in seconds

def _make_one_case(rng, scanner_gain, scanner_offset, mix):
    """Return (pre, post, mask) 3D arrays for one synthetic patient.
    `mix` = (p_uptake, p_plateau, p_washout) proportions for this tumour."""
    X, Y, Z = VOL_SHAPE
    xx, yy, zz = np.mgrid[0:X, 0:Y, 0:Z]

    # background breast tissue: low signal + noise
    pre = rng.normal(40, 5, size=VOL_SHAPE)

    # a spherical tumour at a slightly random location/size
    cx, cy, cz = X//2 + rng.integers(-6,6), Y//2 + rng.integers(-6,6), Z//2
    radius     = rng.integers(7, 11)
    dist       = np.sqrt((xx-cx)**2 + (yy-cy)**2 + ((zz-cz)*2.0)**2)
    mask       = (dist < radius).astype(np.uint8)

    # tumour baseline a bit brighter than background
    pre[mask == 1] += rng.normal(25, 4, size=int(mask.sum()))

    # post-contrast: enhance the tumour. Each tumour has its OWN composition
    # (`mix`) so per-case pie charts genuinely differ.
    post = pre.copy()
    tumour_idx = np.where(mask == 1)
    n = len(tumour_idx[0])
    pattern = rng.choice(3, size=n, p=mix)            # 0=uptake 1=plateau 2=washout
    delta = np.empty(n)
    delta[pattern == 0] = rng.normal( 0.45, 0.05, (pattern==0).sum())  # +45% strong rise
    delta[pattern == 1] = rng.normal( 0.05, 0.03, (pattern==1).sum())  # ~+5% flat
    delta[pattern == 2] = rng.normal(-0.15, 0.04, (pattern==2).sum())  # -15% washout
    post[tumour_idx] = pre[tumour_idx] * (1.0 + delta)

    # ---- SCANNER / HOSPITAL effect (the batch effect) ----
    pre  = pre  * scanner_gain + scanner_offset
    post = post * scanner_gain + scanner_offset
    return pre.astype(np.float32), post.astype(np.float32), mask

def build_synthetic_dataset(root=DATA_ROOT):
    # each center gets a different scanner gain & offset (intensity batch
    # effect) AND a slightly different tumour-composition tendency.
    scanner = {"CENTER_A": (1.00,  0.0),
               "CENTER_B": (1.35, 30.0)}      # B reads ~35% higher + a bias
    # Dirichlet alphas -> per-case mixes vary a lot; centers lean differently
    comp_alpha = {"CENTER_A": (3.0, 2.0, 1.2),   # A: more uptake-dominant tumours
                  "CENTER_B": (1.4, 2.0, 3.0)}   # B: more washout-dominant tumours
    affine = np.diag([1.0, 1.0, 1.0, 1.0])    # 1x1x1 mm voxels
    for center in CENTERS:
        seg_dir = os.path.join(root, center, "segment")
        os.makedirs(seg_dir, exist_ok=True)
        gain, offset = scanner[center]
        rng = np.random.default_rng(hash(center) % (2**32))
        for k in range(1, N_CASES + 1):
            cid = f"{center}_{k:03d}"
            cdir = os.path.join(root, center, cid)
            os.makedirs(cdir, exist_ok=True)
            mix = rng.dirichlet(comp_alpha[center])     # this tumour's composition
            pre, post, mask = _make_one_case(rng, gain, offset, mix)
            nib.save(nib.Nifti1Image(pre,  affine), os.path.join(cdir, f"{cid}_0000.nii.gz"))
            nib.save(nib.Nifti1Image(post, affine), os.path.join(cdir, f"{cid}_0001.nii.gz"))
            nib.save(nib.Nifti1Image(mask, affine), os.path.join(seg_dir, f"{cid}.nii.gz"))
    return root

build_synthetic_dataset()
print("Synthetic dataset created under:", DATA_ROOT)
for f in sorted(glob.glob(os.path.join(DATA_ROOT, "*", "*", "*.nii.gz")))[:6]:
    print("  ", f)
print("   ... (8 cases x 3 files total)")

---
## Part 1 — Reading & Exploring a NIfTI Image

🎓 **Learning goals:** open a `.nii.gz` file, understand what's inside it, and look at a slice.

### Concept — what is a NIfTI volume?

A brain/breast MRI is a **3D grid of numbers**. `.nii.gz` (NIfTI) is the standard file format
to store that grid plus metadata. Two things matter for us:

* **`get_fdata()`** → the raw 3D `numpy` array of intensities (`shape = (X, Y, Z)`).
* the **header / affine** → voxel size, orientation. We mostly keep it untouched and re-use it
  when we save results, so our outputs line up with the input.

> 💡 **Analogy.** A NIfTI file is like a stack of greyscale photos (the *slices*), plus a sticky
> note (the *header*) saying how big each pixel is in millimetres.

### 🎯 Task 1 — write `load_nifti` and `describe_volume`

`load_nifti(path)` returns `(data_array, nifti_object)`.
`describe_volume(name, data)` prints shape, dtype, and intensity range — the first thing any
medical-imaging engineer does to **sanity-check** a file (this mirrors the original
`explore_nifti.py`).

In [ ]:
def load_nifti(path):
    """Load a .nii.gz file. Returns (data: np.ndarray, img: nib object)."""
    # TODO (Part 1): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("load_nifti() is not implemented yet — Part 1")

def describe_volume(name, data):
    """Print a quick quality-control summary of a 3D volume."""
    # TODO (Part 1): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("describe_volume() is not implemented yet — Part 1")

# Try it on the first case of CENTER_A
case_dir = os.path.join(DATA_ROOT, "CENTER_A", "CENTER_A_001")
pre,  _   = load_nifti(os.path.join(case_dir, "CENTER_A_001_0000.nii.gz"))
post, _   = load_nifti(os.path.join(case_dir, "CENTER_A_001_0001.nii.gz"))
mask, _   = load_nifti(os.path.join(DATA_ROOT, "CENTER_A", "segment", "CENTER_A_001.nii.gz"))

describe_volume("pre-contrast  (_0000)", pre)
describe_volume("post-contrast (_0001)", post)
describe_volume("tumour mask",           mask)
print("\nUnique values in mask:", np.unique(mask), " <- binary: 0=background, 1=tumour")

### See it — visualise one slice

Medical images are 3D, but screens are 2D, so we look at one **slice** at a time. We pick the
slice that contains the *most* tumour, then show pre, post, and the mask overlaid.

In [ ]:
def best_slice(mask):
    """Return the z-index of the slice with the largest tumour area."""
    areas = mask.reshape(-1, mask.shape[2]).sum(axis=0)  # voxels per slice
    return int(np.argmax(areas))

z = best_slice(mask)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(pre[:, :, z].T,  cmap="gray", origin="lower"); ax[0].set_title(f"Pre-contrast (z={z})")
ax[1].imshow(post[:, :, z].T, cmap="gray", origin="lower"); ax[1].set_title("Post-contrast")
ax[2].imshow(post[:, :, z].T, cmap="gray", origin="lower")
ax[2].imshow(np.ma.masked_where(mask[:, :, z]==0, mask[:, :, z]).T,
             cmap="autumn", origin="lower", alpha=0.6)
ax[2].set_title("Tumour ROI overlay")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

### ✏️ Exercises — Part 1

1. The tumour looks brighter on the post-contrast image than on the pre. Where in the displayed
   summary numbers can you *already* see that enhancement, before doing any analysis?
2. Write a one-liner that prints how many voxels are tumour in this case.

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 2 — Kinetic Analysis: Classifying Enhancement Patterns

🎓 **Learning goals:** turn *two* volumes into *one* map that labels every tumour voxel as
**uptake / plateau / washout**, and summarise it into numbers.

### Concept — percent enhancement

For each voxel we compute how much it changed from pre to post, **as a percentage**:

$$\Delta SI\,[\%] \;=\; \frac{S_{post}-S_{pre}}{S_{pre}+\varepsilon}\times 100$$

The tiny $\varepsilon$ (`1e-10`) just prevents division by zero. Then we apply thresholds:

| Class | Code | Rule (this pipeline) |
|---|---|---|
| Uptake  | 1 | $\Delta SI > 15\%$ |
| Plateau | 2 | $-5\% \le \Delta SI \le 15\%$ |
| Washout | 3 | $\Delta SI < -5\%$ |
| Background | 0 | outside the ROI |

> ⚠️ **Real-world lesson.** The project's README *text* says "10% / ~10% / 10%" but the actual
> **code** uses 15 / −5. When documentation and code disagree, **the code is the ground truth**
> for what was really computed. Always check. We follow the code.

### 🎯 Task 2 — write `percent_enhancement` and `classify_kinetics`

In [ ]:
def percent_enhancement(pre, post, eps=1e-10):
    """Voxel-wise percent signal change from pre to post contrast."""
    # TODO (Part 2): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("percent_enhancement() is not implemented yet — Part 2")

def classify_kinetics(change, roi, up_thr=15, plat_lo=-5, plat_hi=15):
    """
    Build the colormap of kinetic classes (uint8):
        0 = background, 1 = uptake, 2 = plateau, 3 = washout
    Only voxels inside `roi` (boolean) are classified.
    """
    # TODO (Part 2): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("classify_kinetics() is not implemented yet — Part 2")

roi    = mask > 0
change = percent_enhancement(pre, post)
colormap = classify_kinetics(change, roi)

print("Voxels per class inside the tumour:")
for v, name in [(1,"Uptake"), (2,"Plateau"), (3,"Washout")]:
    print(f"  {name:8s}: {(colormap==v).sum():5d}")

### 🎯 Task 2b — summarise the map into biomarkers

A radiologist does not want a million voxels — they want a few numbers per patient. The most
important: **what fraction of the tumour is uptake / plateau / washout**, plus simple intensity
statistics. These percentages are the headline biomarkers used later for harmonization.

In [ ]:
def kinetic_features(pre, post, mask):
    """Return (features dict, colormap) for one case. Mirrors the original pipeline."""
    # TODO (Part 2): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("kinetic_features() is not implemented yet — Part 2")

feats, cmap = kinetic_features(pre, post, mask)
for k, v in feats.items():
    print(f"  {k:28s} = {v:.2f}")

### ✏️ Exercises — Part 2

1. The three percentages should add up to ~100%. Verify it in code. Why *exactly* 100 here?
2. Re-run `classify_kinetics` with a stricter uptake threshold `up_thr=30`. What happens to the
   uptake vs plateau percentages, and why?

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 3 — Pseudo-color Map Visualization

🎓 **Learning goal:** turn the class map into the colour picture clinicians actually look at.

### Concept

Numbers don't help a radiologist at a glance — **colour does**. We draw the anatomical image in
grey, then paint a semi-transparent layer on top: **blue = uptake, green = plateau,
red = washout**. Red regions (washout) are the ones that worry oncologists, so they must
"pop out".

This reproduces the `*_complete_colormap.png` output of the original `complete_pipeline.py`.

### 🎯 Task 3 — write `plot_colormap`

In [ ]:
# 0=transparent background, 1=blue, 2=green, 3=red  (with alpha)
KINETIC_CMAP = ListedColormap([(0,0,0,0), (0,0,1,0.7), (0,1,0,0.7), (1,0,0,0.7)])

def plot_colormap(case_id, pre, colormap, mask, ax=None):
    """Overlay the kinetic class map on the grey anatomical image."""
    # TODO (Part 3): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("plot_colormap() is not implemented yet — Part 3")

plot_colormap("CENTER_A_001", pre, cmap, mask)

### ✏️ Exercise — Part 3

Show a **montage** of every tumour-containing slice for this case (one subplot per slice) using
`plot_colormap` and a grid of axes.

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 4 — Exporting an RGB NIfTI for Medical Viewers

🎓 **Learning goal:** save the colormap as a file clinical software (Mango, ITK-SNAP, 3D Slicer)
can open in colour.

### Concept

Our `colormap` array holds class *codes* (0–3). A radiologist's viewer doesn't know "2 means
plateau". The trick the original `rgb_nifti_converter.py` uses:

1. normalise the grey anatomical image to 0–255 → use it as a greyscale background in **R, G,
   B** channels;
2. paint class colours on top (blue / green / red);
3. store it with NIfTI's special **RGB datatype (code 128, 24-bit)** using a *structured*
   numpy dtype with named fields `('R','G','B')`.

> 💡 **Why a structured dtype?** A normal `(X,Y,Z,3)` array would be read as a *4D* volume.
> The `('R','G','B')` record dtype tells NIfTI "this is **one** 3D volume whose voxels are
> colours", which is what viewers expect.

### 🎯 Task 4 — write `save_rgb_nifti`

In [ ]:
def save_rgb_nifti(ref_path, class_arr, out_path):
    """
    Save a class map as an RGB-encoded NIfTI (datatype 128) so medical
    viewers show it in colour over a greyscale anatomical background.
    Adapted from the project's rgb_nifti_converter.py.
    """
    # TODO (Part 4): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("save_rgb_nifti() is not implemented yet — Part 4")

ref_path = os.path.join(case_dir, "CENTER_A_001_0000.nii.gz")
out_path = os.path.join(case_dir, "CENTER_A_001_colormap.nii.gz")
save_rgb_nifti(ref_path, cmap, out_path)

# verify by re-loading
back = nib.load(out_path)
print("Saved :", out_path)
print("dtype :", back.get_data_dtype(), "| NIfTI datatype code:", int(back.header["datatype"]))
print("A washout-painted voxel reads as RGB:",
      np.asarray(back.dataobj)[tuple(np.argwhere(cmap == 3)[0])])

### ✏️ Exercise — Part 4

If you skipped the structured-dtype step and saved the plain `(X, Y, Z, 3)` array instead, how
would a viewer interpret the file, and why is that wrong?

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 5 — Radiomics Features

🎓 **Learning goal:** describe the tumour with a vector of numbers ("**radiomics**") instead of
just looking at it.

### Concept

**Radiomics** = extracting many quantitative descriptors from a region of interest, so that
statistics / machine learning can use the image. Categories:

* **First-order** — the intensity *histogram*: mean, std, skewness, kurtosis, percentiles,
  entropy. (We implement these by hand — it's good signal-processing practice.)
* **Shape** — volume, surface, sphericity.
* **Texture** — spatial patterns (GLCM, GLRLM, …).

The original pipeline uses the **PyRadiomics** library for the full set. PyRadiomics is heavy
and not always installed, so here we build the **first-order** features ourselves and treat
PyRadiomics as an *optional* extra.

### 🎯 Task 5 — write `first_order_features`

In [ ]:
def first_order_features(values, prefix):
    """First-order (histogram-based) radiomics features for a 1D array of ROI intensities."""
    # TODO (Part 5): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("first_order_features() is not implemented yet — Part 5")

def radiomics_features(pre, post, mask):
    """First-order radiomics at both timepoints + their temporal change."""
    # TODO (Part 5): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("radiomics_features() is not implemented yet — Part 5")

rf = radiomics_features(pre, post, mask)
for k, v in rf.items():
    print(f"  {k:22s} = {v:8.3f}")

<details>
<summary><b>Optional: the real PyRadiomics (click — runs only if installed)</b></summary>

The original pipeline calls PyRadiomics like below. It needs `SimpleITK` + `pyradiomics`
(`pip install SimpleITK pyradiomics`). This cell is safe to run even if they're missing.
</details>

In [ ]:
try:
    import SimpleITK as sitk
    from radiomics.featureextractor import RadiomicsFeatureExtractor
    ex = RadiomicsFeatureExtractor(binWidth=5, normalize=False,
                                   resampledPixelSpacing=[1, 1, 1])
    img  = sitk.ReadImage(ref_path)
    msk  = sitk.ReadImage(os.path.join(DATA_ROOT,"CENTER_A","segment","CENTER_A_001.nii.gz"))
    msk  = sitk.Resample(msk, img, sitk.Transform(), sitk.sitkNearestNeighbor)
    res  = ex.execute(img, msk, label=1)
    keys = [k for k in res if k.startswith("original_firstorder")][:5]
    print("PyRadiomics is available. Example features:")
    for k in keys:
        print(f"  {k} = {float(res[k]):.3f}")
except Exception as e:
    print("PyRadiomics not available in this environment — that's fine for the lab.")
    print("We use our hand-written first_order_features() instead.")
    print(f"(detail: {type(e).__name__})")

### ✏️ Exercise — Part 5

`skewness` measures asymmetry of the intensity histogram. Without computing it, predict the
**sign** of `t1_skewness` for a tumour where a small bright sub-region enhances strongly, then
check.

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 6 — Multi-Center Harmonization

🎓 **Learning goal:** understand *why* pooling hospitals naively is wrong, and remove the
scanner batch effect.

### Concept — batch effects

We deliberately built `CENTER_B` with a scanner that reads **~35% higher with a +30 offset**.
So `CENTER_B`'s `t1_mean_intensity` is systematically larger than `CENTER_A`'s — *not* because
its tumours differ, but because of the **machine**. If you train a model on pooled data, it may
learn "high intensity ⇒ center B" instead of real biology. That contamination is a **batch
effect**.

Two tools, applied to the **feature table** (rows = patients, columns = features):

1. **Normalization** (e.g. min–max to 0–1): puts features on a common scale.
2. **Harmonization (ComBat)**: explicitly *removes the center-specific location & scale* while
   preserving the real patient-to-patient (biological) variation. ComBat models each feature
   as `value = grand_mean + center_shift + center_scale·noise` and uses *empirical Bayes* to
   estimate the center terms robustly, then subtracts them.

We implement a **teaching-grade location/scale ComBat** (the core idea, no empirical-Bayes
shrinkage) and point to the production `neuroCombat` used in the original code.

### 🎯 Task 6a — build the multi-center feature table

First we need a table. We loop over **every case in every center** and stack the kinetic +
radiomics features. (This is the heart of `process_case`, which Part 7 will formalise.)

In [ ]:
def feature_table(root=DATA_ROOT):
    # TODO (Part 6): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("feature_table() is not implemented yet — Part 6")

raw_df = feature_table()
print(f"Feature table: {raw_df.shape[0]} patients x {raw_df.shape[1]-2} features\n")
print(raw_df.groupby("center")[["t1_mean_intensity","uptake_percentage"]]
            .agg(["mean","std"]).round(2))

Look at `t1_mean_intensity` above: `CENTER_B`'s mean is far higher than `CENTER_A`'s — exactly
the scanner batch effect we injected. That is what harmonization must remove.

### 🎯 Task 6b — normalize, then harmonize

In [ ]:
def minmax_normalize(df, feature_cols):
    # TODO (Part 6): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("minmax_normalize() is not implemented yet — Part 6")

def simple_combat(df, feature_cols, batch_col="center"):
    """
    Teaching-grade location/scale harmonization (the core of ComBat without
    the empirical-Bayes shrinkage). For every feature, every center is
    re-centred and re-scaled to the pooled (grand) mean and std.
    """
    # TODO (Part 6): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("simple_combat() is not implemented yet — Part 6")

feat_cols = [c for c in raw_df.columns if c not in ("case_id", "center")]
norm_df   = minmax_normalize(raw_df, feat_cols)
harm_df   = simple_combat(norm_df, feat_cols)

def between_center_var(df, col):
    """Variance of the per-center means = how much centers disagree."""
    return df.groupby("center")[col].mean().var()

print("Between-center variance (lower = better harmonized):\n")
print(f"{'feature':28s} {'raw':>10s} {'normalized':>12s} {'harmonized':>12s}")
for col in ["t1_mean_intensity", "uptake_percentage", "washout_percentage"]:
    print(f"{col:28s} {between_center_var(raw_df,col):10.4f} "
          f"{between_center_var(norm_df,col):12.4f} "
          f"{between_center_var(harm_df,col):12.6f}")

The harmonized between-center variance collapses toward ~0: the systematic hospital difference
is gone, while patient-to-patient differences within each center are preserved.

<details>
<summary><b>Production note: neuroCombat (what the original code uses)</b></summary>

The real `complete_pipeline.py` calls the `neuroCombat` library:

```python
from neuroCombat import neuroCombat
harmonized = neuroCombat(dat=data_matrix,            # features x samples
                         batch=center_labels,
                         mod=None, par_prior=True, eb=True)["data"]
```

`eb=True` turns on **empirical-Bayes shrinkage**, which makes the per-center estimates robust
when you have few samples per center — important with only a handful of patients. Our
`simple_combat` is the same idea without that shrinkage, so it's easier to read.
</details>

### ✏️ Exercise — Part 6

Why do we harmonize the **feature table** and not the raw images directly? Give one practical
reason.

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 7 — Putting It All Together: the Pipeline

🎓 **Learning goal:** assemble every function you wrote into the single class the original
project ships, and produce the same outputs (per-case colormaps + 3 CSV files).

Notice there is **almost no new code** here — Part 7 just *orchestrates* Parts 1–6. That is the
whole point of building piece by piece.

In [ ]:
class DCEMRIPipeline:
    """End-to-end pipeline, assembled from the functions built in Parts 1-6."""

    def process_case(self, cid, case_dir, seg_dir):
        # TODO (Part 7): implement this function — see the Task description above.
        # (delete this line once done)
        raise NotImplementedError("process_case() is not implemented yet — Part 7")

    def run(self, root=DATA_ROOT):
        # TODO (Part 7): implement this function — see the Task description above.
        # (delete this line once done)
        raise NotImplementedError("run() is not implemented yet — Part 7")

pipe = DCEMRIPipeline()
raw, norm, harm = pipe.run()
print(f"\nFinal table: {raw.shape[0]} patients x {raw.shape[1]-1} features")
raw.head()

### ✏️ Exercise — Part 7

Add **one** new biomarker (e.g. `enhancement_ratio = t1_mean / t0_mean`) so it appears in all
three CSVs. Which function do you edit — and how many others?

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 8 — Validation Dashboard

🎓 **Learning goal:** produce the figure that *proves* the pipeline worked, like the original
`combat_visualization.py`: reference kinetic curves + before/after harmonization comparison.

In [ ]:
def reference_curves(ax):
    """Idealised uptake / plateau / washout kinetic curves (teaching figure)."""
    t = np.linspace(0, 3, 100)
    base = 75 * (1 - np.exp(-2.0 * t))           # common early rise
    div  = 1.3                                   # divergence time
    i    = np.argmin(np.abs(t - div))
    up, pl, wo = base.copy(), base.copy(), base.copy()
    for j in range(i+1, len(t)):
        dt = t[j] - t[i]
        up[j] = base[i] + dt*5
        pl[j] = base[i] + dt*(-3)
        wo[j] = base[i] + dt*(-10)
    ax.plot(t, up, color="#3274A1", lw=2, label="Uptake")
    ax.plot(t, pl, color="#3A923A", lw=2, label="Plateau")
    ax.plot(t, wo, color="#C03D3E", lw=2, label="Washout")
    ax.axvline(div, color="gray", ls="--", alpha=.6)
    ax.set(title="Reference kinetic curves", xlabel="Time-point",
           ylabel="Signal intensity (%)", xlim=(0,3), ylim=(0,120))
    ax.legend(); ax.grid(alpha=.3)

def comparison_bars(ax, raw_df, harm_df, feature, title, color):
    centers = sorted(raw_df["center"].unique())
    x = np.arange(len(centers)); w = 0.35
    rm = [raw_df [raw_df ["center"]==c][feature].mean() for c in centers]
    hm = [harm_df[harm_df["center"]==c][feature].mean() for c in centers]
    ax.bar(x-w/2, rm, w, label="Raw",        color=color, alpha=.85)
    ax.bar(x+w/2, hm, w, label="Harmonized", color=color, alpha=.40)
    ax.set(title=title, xticks=x); ax.set_xticklabels(centers)
    ax.legend(); ax.grid(axis="y", alpha=.3)

raw_c  = raw.copy();  raw_c["center"]  = raw_c["case_id"].str.rsplit("_",n=1).str[0]
harm_c = harm.copy(); harm_c["center"] = harm_c["case_id"].str.rsplit("_",n=1).str[0]

fig = plt.figure(figsize=(15, 4.5))
reference_curves(fig.add_subplot(1, 3, 1))
comparison_bars(fig.add_subplot(1, 3, 2), raw_c, harm_c,
                "t1_mean_intensity", "t1 mean intensity: raw vs harmonized", "#1f77b4")
comparison_bars(fig.add_subplot(1, 3, 3), raw_c, harm_c,
                "uptake_percentage", "Uptake %: raw vs harmonized", "#2ca02c")
plt.tight_layout(); plt.show()

In the middle panel the two **raw** bars are very different heights (scanner batch effect);
after **harmonization** the bars match — the centers now agree. That single picture is the
"proof it worked" you put in a paper or a report.

### ✏️ Exercise — Part 8

Add a 4th panel comparing `washout_percentage`. Does harmonization change it as much as
`t1_mean_intensity`? Explain why ratio-type features are *naturally* more robust to scanner
gain than absolute-intensity features.

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## Part 9 — Descriptive Statistics & Composition Pie Charts

🎓 **Learning goal:** summarise the whole cohort with numbers and the right pictures —
per-case **composition pie charts** and per-center **descriptive statistics** (this reproduces
the *signal-distribution pie charts* and *key-metrics tables* from the original web app).

### Concept — describing a cohort

After Part 7 we have one row per patient. Before any modelling, you always **describe** the
data: central tendency (`mean`, `median`), spread (`std`, min–max), and *how it differs by
center* (the batch-effect story again, now in table form).

### Concept — when is a pie chart the right chart?

A pie shows **parts of a single whole**. The kinetic percentages
(uptake + plateau + washout ≈ 100%) of **one tumour** are exactly that — a perfect pie.

> ⚠️ **Good practice.** Pies are *bad* for comparing across patients or centers (humans compare
> angles poorly). For comparisons use grouped **bars** or **boxplots**. Rule of thumb:
> *pie = composition of one thing; bar/box = comparison between things.* You will use both below.

### 🎯 Task 9a — write `kinetic_pie`

Draw the uptake / plateau / washout composition of **one case** as a pie, using the project's
colour convention (**blue = uptake, green = plateau, red = washout**, same as Part 3).

In [ ]:
# project-wide kinetic colours (match the Part 3 overlay)
KIN_LABELS = ["Uptake", "Plateau", "Washout"]
KIN_COLS   = ["uptake_percentage", "plateau_percentage", "washout_percentage"]
KIN_HEX    = ["#1f77b4", "#2ca02c", "#d62728"]   # blue / green / red

def kinetic_pie(row, ax=None):
    """Pie chart of one case's kinetic composition. `row` is a feature-table row."""
    # TODO (Part 9): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("kinetic_pie() is not implemented yet — Part 9")

# show the composition of the first case
kinetic_pie(raw.iloc[0])

### 🎯 Task 9b — write `kinetic_summary`

Return a tidy **descriptive-statistics table** of the three kinetic percentages, **grouped by
center** (so the per-site differences are visible as numbers, not just figures). Recover the
center from `case_id` *inside* the function so it works on any feature table.

In [ ]:
def kinetic_summary(df):
    """Per-center descriptive statistics of the kinetic percentages."""
    # TODO (Part 9): implement this function — see the Task description above.
    # (delete this line once done)
    raise NotImplementedError("kinetic_summary() is not implemented yet — Part 9")

print("Per-center descriptive statistics (raw features):\n")
print(kinetic_summary(raw))

print("\nWhole-cohort summary (raw features):\n")
print(raw[KIN_COLS].describe().round(2))

### See it — cohort composition grid + distribution boxplots

This is a *demonstration* cell (visualization plumbing, given for you). It uses your
`kinetic_pie` for a pie-per-patient grid, then **boxplots** to compare the distribution of each
kinetic class across centers — the comparison the pies cannot show well.

In [ ]:
cases = raw.reset_index(drop=True)
n = len(cases); cols = 4; rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 3.2*rows))
for ax, (_, r) in zip(axes.ravel(), cases.iterrows()):
    kinetic_pie(r, ax=ax)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.suptitle("Per-case kinetic composition", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

# distribution comparison across centers (the right chart for comparison)
cmp = raw.copy()
cmp["center"] = cmp["case_id"].str.rsplit("_", n=1).str[0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, hexc in zip(axes, KIN_COLS, KIN_HEX):
    groups = [cmp.loc[cmp.center == c, col].values for c in sorted(cmp.center.unique())]
    bp = ax.boxplot(groups, labels=sorted(cmp.center.unique()), patch_artist=True)
    for box in bp["boxes"]:
        box.set(facecolor=hexc, alpha=0.5)
    ax.set_title(col.replace("_", " ")); ax.set_ylabel("%"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

### ✏️ Exercises — Part 9

1. From the per-center table, which kinetic class differs **most** between `CENTER_A` and
   `CENTER_B` in the *raw* features? Is it still different after harmonization
   (`kinetic_summary(harm)`)?
2. Add a **coefficient of variation** (`std / mean`) column to `kinetic_summary`. Why is CV
   sometimes more informative than std alone?
3. You have a pie per patient. Why would a single pie of the *cohort-average* uptake/plateau/
   washout be **misleading** for a clinician?

> 💡 *Try this yourself first. The worked solution is in the instructor notebook.*

---
## 🎉 Wrap-up

You built, piece by piece, a complete multi-center DCE-MRI pipeline:

| Part | You can now… |
|---|---|
| 1 | load & QC a NIfTI volume |
| 2 | classify enhancement kinetics (uptake/plateau/washout) |
| 3 | make the clinical pseudo-color overlay |
| 4 | export an RGB NIfTI for medical viewers |
| 5 | extract first-order radiomics features |
| 6 | normalize **and** harmonize multi-center features |
| 7 | orchestrate everything into one pipeline + CSV outputs |
| 8 | produce the validation dashboard |
| 9 | describe the cohort: composition pie charts + per-center statistics |

### Mini-glossary

- **DCE-MRI** — MRI series before/after a contrast agent injection.
- **ROI / mask** — voxels marked as tumour by a radiologist.
- **Kinetic pattern** — uptake / plateau / washout shape of the enhancement.
- **Radiomics** — many quantitative features extracted from the ROI.
- **Batch effect** — systematic difference caused by scanner/site, not biology.
- **ComBat / harmonization** — statistical removal of batch effects from features.

### From this lab to real research

Everything here runs on the **real MAMA-MIA dataset** with *no code changes*: replace the
`build_synthetic_dataset()` call by pointing `DATA_ROOT` at the real folders (same
`CENTER/CASE/..._0000/_0001` + `segment/` layout), and swap `simple_combat` for `neuroCombat`
(`pip install neuroCombat`) for the empirical-Bayes version. The original project also wraps
all of this in a Flask web app for clinicians — a natural extension once this lab is mastered.

